# UPI Fraud Detection using LSTM (RNN)

## Project Overview

India's **Unified Payments Interface (UPI)** has grown into one of the world's largest real-time payment ecosystems, processing over 100 billion transactions annually across hundreds of millions of users. As transaction volumes scale, so does the surface area for fraud — account takeovers, social-engineering scams, and high-velocity anomalous transfers now represent a significant challenge for NPCI and member banks.

### Why Sequential Models?

Fraud rarely manifests in a single isolated transaction. Instead, it unfolds as a **temporal pattern**: a sequence of seemingly normal activity followed by a device change, a burst of rapid transfers to unfamiliar payees, and finally large late-night withdrawals. Traditional ML classifiers (logistic regression, random forest) treat each transaction independently and miss these cross-transaction signals.

**Long Short-Term Memory (LSTM)** networks are a class of Recurrent Neural Networks (RNN) specifically designed to learn patterns over variable-length sequences. By reading a window of 15 consecutive UPI transactions per user session, the LSTM can:

- Learn the *order* in which suspicious features escalate
- Capture *velocity* patterns (rapid successive transfers)
- Weight recent events more heavily than older context
- Generalize across diverse user spending profiles

### Model Architecture

This notebook implements a **2-layer stacked LSTM** with:

| Component | Details |
|---|---|
| Input | 15-step sequence × 12 transaction features |
| LSTM Layer 1 | 128 hidden units, returns full sequence |
| LSTM Layer 2 | 128 hidden units, returns last hidden state |
| Classifier Head | Dropout → Linear(128→64) → BatchNorm → ReLU → Dropout → Linear(64→1) |
| Output | Binary: Fraudulent (1) / Legitimate (0) |
| Loss | BCEWithLogitsLoss with class-imbalance pos_weight |

### Data

Synthetic UPI session data is generated and calibrated to **official NPCI monthly statistics** (FY 2021-22, 2022-23, 2023-24). The calibration sets the average per-transaction amount to match real-world UPI averages, making the synthetic dataset statistically grounded.

- **80,000 sessions** — 23% fraudulent, 77% legitimate
- **12 features** per transaction step: amount, hour of day, day of week, weekend flag, late-night flag, new payee flag, device changed flag, high-amount flag, transaction velocity, time since last transaction, cumulative amount ratio, payee familiarity
- **Sequence length**: 15 transactions per session

In [ ]:
# Install required dependencies
!pip install torch pandas numpy scikit-learn matplotlib seaborn openpyxl

## Step 1: Upload NPCI Data Files

This notebook uses **official NPCI monthly UPI statistics** to calibrate the synthetic data generator. Specifically, it reads the average transaction value (Value in Crore / Volume in Million) from three fiscal year Excel files:

- `Product-Statistics-UPI-Upi-monthly-statistics-2021-22-monthly.xlsx`
- `Product-Statistics-UPI-Upi-monthly-statistics-2022-23-monthly.xlsx`
- `Product-Statistics-UPI-Upi-monthly-statistics-2023-24-monthly.xlsx`

These files are available from [NPCI's official statistics page](https://www.npci.org.in/what-we-do/upi/upi-ecosystem-statistics).

**If the files are not available**, the notebook will automatically fall back to a hardcoded average of **Rs. 1,650.00** per transaction (a reasonable estimate based on published UPI statistics).

> Run the cell below and select all three Excel files when the upload widget appears. You can skip the upload and re-run if you want to use the fallback value.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
except Exception:
    print("Not in Colab or upload skipped — will use fallback avg value.")

## Step 2: Data Preparation (Synthetic UPI Sequences)

This step performs the following operations:

1. **Load & parse NPCI Excel files** — Extract monthly volume and value data. Compute the calibrated average per-transaction value in INR. Falls back to Rs. 1,650 if files are unavailable.

2. **Generate synthetic UPI sessions** — 80,000 sessions (23% fraud, 77% legitimate). Each session is a sequence of 15 transactions described by 12 behavioral features.

3. **Legitimate session generation** — Transactions drawn from user-specific lognormal amount distributions, with realistic hour/day patterns. Small random noise injected for device changes and occasional high amounts.

4. **Fraudulent session generation** — Follows a realistic attack pattern:
   - First 8 steps: normal-looking activity (reconnaissance)
   - Step 9: device change event
   - Steps 10-12: escalating amounts to new payees, high velocity
   - Steps 13-14: large late-night transfers (3x–5x user average)
   - Step 15: final large drain transfer

5. **Label noise** — 7% of labels are randomly flipped to simulate annotation noise and improve model robustness.

6. **Scaling** — StandardScaler applied across the flattened feature dimension.

7. **Train/test split** — 80/20 stratified split.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
rng = np.random.default_rng(42)

# ---------------------------------------------------------------------------
# Load & Parse NPCI Monthly Statistics (with fallback)
# ---------------------------------------------------------------------------
print("Loading NPCI monthly UPI statistics...")

avg_txn_value_inr = None
try:
    monthly_records = []
    for yr_tag in ['2021-22', '2022-23', '2023-24']:
        fpath = f'Product-Statistics-UPI-Upi-monthly-statistics-{yr_tag}-monthly.xlsx'
        raw = pd.read_excel(fpath, header=None)
        raw.columns = ['Month', 'Volume_Mn', 'Avg_Daily_Volume_Mn', 'Value_Cr', 'Avg_Daily_Value_Cr']
        raw = raw.iloc[1:].dropna(subset=['Month']).reset_index(drop=True)
        monthly_records.append(raw)

    monthly_df = pd.concat(monthly_records, ignore_index=True)
    for col in ['Volume_Mn', 'Value_Cr']:
        monthly_df[col] = pd.to_numeric(monthly_df[col], errors='coerce')
    monthly_df = monthly_df.dropna()

    avg_txn_value_inr = (
        (monthly_df['Value_Cr'].mean() * 1e7) /
        (monthly_df['Volume_Mn'].mean() * 1e6)
    )
    print(f"Loaded {len(monthly_df)} monthly records.")
    print(f"Calibrated avg per-transaction amount: Rs.{avg_txn_value_inr:.2f}")
except Exception as e:
    avg_txn_value_inr = 1650.0
    print(f"NPCI files not found or could not be parsed ({e}).")
    print(f"Using fallback avg per-transaction amount: Rs.{avg_txn_value_inr:.2f}")

print()

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
N_SESSIONS = 80_000
FRAUD_RATE  = 0.23
SEQ_LEN     = 15
FEATURE_COLS = [
    'amount', 'hour_of_day', 'day_of_week', 'is_weekend',
    'is_late_night', 'is_new_payee', 'device_changed',
    'high_amount_flag', 'txn_velocity', 'time_since_last',
    'cumulative_amount_ratio', 'payee_familiarity',
]
N_FEATURES = len(FEATURE_COLS)

n_fraud  = int(N_SESSIONS * FRAUD_RATE)
n_normal = N_SESSIONS - n_fraud

print(f"Generating {N_SESSIONS:,} user sessions ({n_fraud:,} fraud, {n_normal:,} normal)...")

mu_log    = np.log(avg_txn_value_inr) - 0.5
sigma_log = 1.3

# ---------------------------------------------------------------------------
# Transaction generators
# ---------------------------------------------------------------------------
def normal_txn(t, user_avg_amount, rng):
    amount   = rng.lognormal(mean=np.log(user_avg_amount) - 0.1, sigma=0.5)
    amount   = float(np.clip(amount, 10, 30_000))
    hour     = int(rng.integers(7, 22))
    dow      = int(rng.integers(0, 7))
    weekend  = int(dow >= 5)
    late     = 0
    new_pay  = int(rng.random() < 0.05)
    dev_chg  = int(rng.random() < 0.01)
    high_amt = int(amount > 50_000)
    velocity = int(rng.integers(1, 4))
    since    = float(rng.exponential(scale=300))
    since    = float(np.clip(since, 10, 1440))
    cum_rat  = float(amount / user_avg_amount)
    payee_f  = 1.0 - new_pay
    return [amount, hour, dow, weekend, late, new_pay, dev_chg, high_amt,
            velocity, since, cum_rat, payee_f]


def fraud_txn_sequence(user_avg_amount, rng):
    seq = []
    # Steps 0-7: normal reconnaissance activity
    for t in range(8):
        seq.append(normal_txn(t, user_avg_amount, rng))
    # Step 8: device change event
    a = rng.lognormal(mean=np.log(user_avg_amount), sigma=0.3)
    seq.append([
        float(np.clip(a, 10, 30_000)),
        int(rng.integers(7, 22)), int(rng.integers(0, 7)), 0, 0,
        0, 1, 0, int(rng.integers(1, 3)), float(rng.exponential(300)),
        float(a / user_avg_amount), 1.0
    ])
    # Steps 9-11: escalating amounts, new payees, high velocity
    for t in range(3):
        a = rng.lognormal(mean=np.log(user_avg_amount) + 0.3, sigma=0.4)
        seq.append([
            float(np.clip(a, 10, 40_000)),
            int(rng.integers(7, 22)), int(rng.integers(0, 7)), 0, 0,
            int(t >= 1), 0, 0,
            int(rng.integers(5, 8)),
            float(rng.uniform(1, 15)),
            float(a / user_avg_amount), float(1 - (t >= 1))
        ])
    # Steps 12-13: large late-night transfers (3x and 5x user average)
    for multiplier in [3, 5]:
        a = user_avg_amount * multiplier * rng.uniform(0.8, 1.2)
        a = float(np.clip(a, 5_000, 4_00_000))
        hour = int(rng.integers(22, 24)) if rng.random() < 0.6 else int(rng.integers(0, 5))
        seq.append([
            a, hour, int(rng.integers(0, 7)), 0, 1,
            1, 0, int(a > 50_000),
            int(rng.integers(6, 10)),
            float(rng.uniform(0.5, 5.0)),
            float(a / user_avg_amount), 0.0
        ])
    # Step 14: final large drain transfer
    a = user_avg_amount * float(rng.uniform(4, 9))
    a = float(np.clip(a, 5_000, 2_00_000))
    is_ln = int(rng.random() < 0.55)
    hour  = int(rng.choice([1, 2, 3, 23])) if is_ln else int(rng.integers(7, 22))
    seq.append([
        a, hour, int(rng.integers(0, 7)), 0, is_ln,
        int(rng.random() < 0.80), 0, int(a > 50_000),
        int(rng.integers(5, 8)),
        float(rng.uniform(0.5, 6.0)),
        float(a / user_avg_amount), 0.0
    ])
    return seq


# ---------------------------------------------------------------------------
# Generate sessions
# ---------------------------------------------------------------------------
all_sequences = []
all_labels    = []

for i in range(n_normal):
    user_avg = float(rng.lognormal(mean=mu_log, sigma=sigma_log))
    user_avg = float(np.clip(user_avg, 100, 20_000))
    seq = [normal_txn(t, user_avg, rng) for t in range(SEQ_LEN)]
    if rng.random() < 0.20:
        t_idx = int(rng.integers(0, SEQ_LEN))
        seq[t_idx][6] = 1
    if rng.random() < 0.15:
        t_idx = int(rng.integers(0, SEQ_LEN))
        seq[t_idx][0] = float(user_avg * rng.uniform(8, 20))
        seq[t_idx][7] = 1
    all_sequences.append(seq)
    all_labels.append(0)

for i in range(n_fraud):
    user_avg = float(rng.lognormal(mean=mu_log, sigma=sigma_log))
    user_avg = float(np.clip(user_avg, 100, 20_000))
    if rng.random() < 0.15:
        seq = [normal_txn(t, user_avg, rng) for t in range(SEQ_LEN)]
        for t in range(SEQ_LEN - 3, SEQ_LEN):
            seq[t][0] = float(user_avg * rng.uniform(3, 8))
            seq[t][5] = 1
            seq[t][8] = int(rng.integers(4, 7))
    else:
        seq = fraud_txn_sequence(user_avg, rng)
    all_sequences.append(seq)
    all_labels.append(1)

# ---------------------------------------------------------------------------
# Build arrays and add noise
# ---------------------------------------------------------------------------
X = np.array(all_sequences, dtype=np.float32)
y = np.array(all_labels,    dtype=np.int32)

noise_std = np.array(
    [0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, 0.3, 0.4, 0.0],
    dtype=np.float32
)
noise = rng.normal(0, 1, size=X.shape).astype(np.float32) * noise_std
X = X + noise
for idx in [3, 4, 5, 6, 7, 11]:
    X[:, :, idx] = np.clip(X[:, :, idx], 0, 1)
X[:, :, 0] = np.maximum(X[:, :, 0], 1.0)

print(f"Dataset: X={X.shape}  |  Fraud rate: {y.mean()*100:.1f}%")

# ---------------------------------------------------------------------------
# Label noise injection
# ---------------------------------------------------------------------------
LABEL_NOISE = 0.07
flip_mask = rng.random(len(y)) < LABEL_NOISE
y = y.copy()
y[flip_mask] = 1 - y[flip_mask]
print(f"After {LABEL_NOISE*100:.0f}% label noise: fraud rate = {y.mean()*100:.1f}%")

# ---------------------------------------------------------------------------
# Scale features and split
# ---------------------------------------------------------------------------
N, T, F = X.shape
X_2d = X.reshape(-1, F)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_2d).reshape(N, T, F).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

# ---------------------------------------------------------------------------
# Save processed data
# ---------------------------------------------------------------------------
os.makedirs('data/processed', exist_ok=True)
np.save('data/processed/X_train.npy', X_train)
np.save('data/processed/X_test.npy',  X_test)
np.save('data/processed/y_train.npy', y_train)
np.save('data/processed/y_test.npy',  y_test)

print("Data preparation complete.")

## Step 3: Build and Train LSTM Model

### Model Design

The `FraudLSTM` architecture is a 2-layer stacked LSTM followed by a classification head:

```
Input (batch, 15, 12)
    └─ LSTM(12 → 128, 2 layers, dropout=0.3)
         └─ last hidden state (batch, 128)
              └─ Dropout(0.3)
                   └─ Linear(128 → 64)
                        └─ BatchNorm1d(64)
                             └─ ReLU
                                  └─ Dropout(0.2)
                                       └─ Linear(64 → 1)  →  logit
```

### Training Details

| Hyperparameter | Value |
|---|---|
| Hidden units | 128 |
| LSTM layers | 2 |
| Dropout | 0.3 (LSTM), 0.2 (classifier) |
| Batch size | 256 |
| Epochs | 20 |
| Optimizer | Adam (lr=1e-3, weight_decay=1e-5) |
| LR scheduler | StepLR (step=5, gamma=0.5) |
| Loss | BCEWithLogitsLoss with pos_weight |
| Gradient clipping | max_norm=1.0 |

### Class Imbalance Handling

With ~23% fraud rate, the model would naively predict everything as legitimate. The `pos_weight` parameter in `BCEWithLogitsLoss` scales the loss for positive (fraud) samples by `min(legit_count / fraud_count, 5.0)`, penalizing missed fraud detections more heavily.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import json

print("PyTorch version:", torch.__version__)

# ---------------------------------------------------------------------------
# Load processed data
# ---------------------------------------------------------------------------
X_train = np.load('data/processed/X_train.npy')
X_test  = np.load('data/processed/X_test.npy')
y_train = np.load('data/processed/y_train.npy')
y_test  = np.load('data/processed/y_test.npy')

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

SEQ_LEN    = X_train.shape[1]
N_FEATURES = X_train.shape[2]

# ---------------------------------------------------------------------------
# Build DataLoaders
# ---------------------------------------------------------------------------
X_tr = torch.tensor(X_train, dtype=torch.float32)
X_te = torch.tensor(X_test,  dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.float32)

BATCH_SIZE   = 256
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=BATCH_SIZE, shuffle=False)

# ---------------------------------------------------------------------------
# Model definition
# ---------------------------------------------------------------------------
class FraudLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super(FraudLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        logit = self.classifier(last_out)
        return logit.squeeze(1)


model = FraudLSTM(input_dim=N_FEATURES, hidden_dim=128, num_layers=2, dropout=0.3)
print(f"\nModel architecture:\n{model}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

# ---------------------------------------------------------------------------
# Loss, optimizer, scheduler
# ---------------------------------------------------------------------------
fraud_count = int(y_train.sum())
legit_count = int(len(y_train) - fraud_count)
raw_weight  = legit_count / fraud_count
pos_weight  = torch.tensor([min(raw_weight, 5.0)], dtype=torch.float32)
print(f"\nPos-weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------
NUM_EPOCHS = 20
history = {'train_loss': [], 'val_loss': []}
print(f"\nStarting training for {NUM_EPOCHS} epochs...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    history['train_loss'].append(epoch_loss)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logits   = model(X_batch)
            loss     = criterion(logits, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(test_loader.dataset)
    history['val_loss'].append(val_loss)
    scheduler.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  Train Loss: {epoch_loss:.4f}  Val Loss: {val_loss:.4f}")

# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
print("\nEvaluating on test set...")
model.eval()
all_logits, all_labels_eval = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        all_logits.extend(logits.cpu().numpy())
        all_labels_eval.extend(y_batch.cpu().numpy())

all_probs  = torch.sigmoid(torch.tensor(all_logits)).numpy()
all_labels_eval = np.array(all_labels_eval, dtype=int)
all_preds  = (all_probs >= 0.5).astype(int)

accuracy  = accuracy_score(all_labels_eval, all_preds)
precision = precision_score(all_labels_eval, all_preds, zero_division=0)
recall    = recall_score(all_labels_eval, all_preds, zero_division=0)
f1        = f1_score(all_labels_eval, all_preds, zero_division=0)
roc_auc   = roc_auc_score(all_labels_eval, all_probs)
cm        = confusion_matrix(all_labels_eval, all_preds)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"\nConfusion Matrix:\n{cm}")

# ---------------------------------------------------------------------------
# Save model and metrics
# ---------------------------------------------------------------------------
os.makedirs('saved_models', exist_ok=True)
torch.save(model.state_dict(), 'saved_models/lstm_model.pth')

metrics = {
    'accuracy':         round(accuracy, 4),
    'precision':        round(precision, 4),
    'recall':           round(recall, 4),
    'f1':               round(f1, 4),
    'roc_auc':          round(roc_auc, 4),
    'confusion_matrix': cm.tolist(),
    'history':          history,
}
with open('saved_models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Model and metrics saved.")

## Step 4: Visualize Results

Four plots are generated to evaluate the trained model:

1. **Training & Validation Loss Curve** — Tracks BCE loss across 20 epochs for both train and validation splits. Convergence without a large gap between the two curves indicates the model is not severely overfitting.

2. **Confusion Matrix** — Shows the absolute counts of True Positives, True Negatives, False Positives, and False Negatives at the 0.5 decision threshold. For fraud detection, the False Negative cell (fraud classified as legitimate) is the most critical metric to minimize.

3. **Performance Metric Bar Chart** — Side-by-side comparison of Accuracy, Precision, Recall, F1-Score, and ROC-AUC on the held-out test set.

4. **ROC Curve** — Plots True Positive Rate (Recall) vs False Positive Rate across all decision thresholds. The Area Under the Curve (AUC) summarizes overall discrimination ability independent of any specific threshold. An AUC of 1.0 is perfect; 0.5 is random.

In [ ]:
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import os

os.makedirs('visualizations', exist_ok=True)

with open('saved_models/metrics.json', 'r') as f:
    metrics = json.load(f)

# ---------------------------------------------------------------------------
# Plot 1: Training & Validation Loss Curve
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
epochs = range(1, len(metrics['history']['train_loss']) + 1)
ax.plot(epochs, metrics['history']['train_loss'], 'b-o', markersize=4, label='Training Loss')
ax.plot(epochs, metrics['history']['val_loss'],   'r-s', markersize=4, label='Validation Loss')
ax.set_title('LSTM Training & Validation Loss\nUPI Fraud Detection', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/loss_curve.png', dpi=150)
plt.show()

# ---------------------------------------------------------------------------
# Plot 2: Confusion Matrix
# ---------------------------------------------------------------------------
cm = np.array(metrics['confusion_matrix'])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Legitimate', 'Fraud'],
    yticklabels=['Legitimate', 'Fraud'],
    ax=ax
)
ax.set_title('Confusion Matrix — LSTM Fraud Classifier', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('visualizations/confusion_matrix.png', dpi=150)
plt.show()

# ---------------------------------------------------------------------------
# Plot 3: Performance Metric Bars
# ---------------------------------------------------------------------------
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
metric_values = [
    metrics['accuracy'], metrics['precision'],
    metrics['recall'], metrics['f1'], metrics['roc_auc']
]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    metric_names, metric_values,
    color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336'],
    width=0.55, edgecolor='white', linewidth=1.2
)
for bar, val in zip(bars, metric_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
        f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
ax.set_ylim(0, 1.12)
ax.set_title('LSTM Fraud Detector — Test Set Performance Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/metric_bars.png', dpi=150)
plt.show()

# ---------------------------------------------------------------------------
# Plot 4: ROC Curve
# (Reload model from disk to demonstrate saved-model portability)
# ---------------------------------------------------------------------------
class FraudLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze(1)


X_test_np  = np.load('data/processed/X_test.npy')
y_test_np  = np.load('data/processed/y_test.npy')
n_features = X_test_np.shape[2]

roc_model = FraudLSTM(input_dim=n_features)
roc_model.load_state_dict(torch.load('saved_models/lstm_model.pth', weights_only=True))
roc_model.eval()

X_te_roc = torch.tensor(X_test_np, dtype=torch.float32)
y_te_roc = torch.tensor(y_test_np, dtype=torch.float32)
roc_loader = DataLoader(TensorDataset(X_te_roc, y_te_roc), batch_size=512, shuffle=False)

all_probs_roc, all_labels_roc = [], []
with torch.no_grad():
    for xb, yb in roc_loader:
        probs = torch.sigmoid(roc_model(xb))
        all_probs_roc.extend(probs.numpy())
        all_labels_roc.extend(yb.numpy())

fpr, tpr, _ = roc_curve(all_labels_roc, all_probs_roc)
roc_auc_val = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'ROC Curve (AUC = {roc_auc_val:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--',
        label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — LSTM UPI Fraud Detector', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/roc_curve.png', dpi=150)
plt.show()

print("All visualizations saved to visualizations/")